# Week 5 — Additional Models: Decision Tree and Random Forest

In this notebook, I tested Decision Tree and Random Forest regression models to predict California residential single-family property close prices. The goal is to compare these tree-based models against the Week 4 Linear Regression baseline.

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [2]:
train_df = pd.read_csv("train_residential_single_family_week3_no_outliers.csv")
test_df = pd.read_csv("test_residential_single_family_week3_no_outliers.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (128568, 57)
Test shape: (11914, 57)


In [3]:
numeric_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "LotSizeAcres",
    "LotSizeArea",
    "YearBuilt",
    "GarageSpaces",
    "ParkingTotal",
    "Latitude",
    "Longitude",
    "Stories",
    "AssociationFee",
    "MainLevelBedrooms"
]

categorical_features = [
    "CountyOrParish",
    "City",
    "PostalCode",
    "HighSchoolDistrict",
    "Flooring",
    "AttachedGarageYN",
    "Levels",
    "PoolPrivateYN",
    "NewConstructionYN",
    "ViewYN",
    "FireplaceYN"
]

In [4]:
reference_only_cols = ["OriginalListPrice", "ListPrice", "DaysOnMarket", "price_per_sqft"]
reference_only_flags = [
    f"{c}_missing_flag" 
    for c in reference_only_cols 
    if f"{c}_missing_flag" in train_df.columns
]

target_derived_cols = ["log_ClosePrice"]

exclude_from_model = reference_only_cols + reference_only_flags + target_derived_cols

feature_numeric = numeric_features.copy()

for col in ["home_age", "log_LivingArea", "log_LotSizeSquareFeet"]:
    if col in train_df.columns:
        feature_numeric.append(col)

missing_flag_cols = [col for col in train_df.columns if col.endswith("_missing_flag")]
feature_numeric += missing_flag_cols

feature_numeric = list(dict.fromkeys([col for col in feature_numeric if col in train_df.columns]))
feature_categorical = [col for col in categorical_features if col in train_df.columns]

feature_numeric = [col for col in feature_numeric if col not in exclude_from_model]
feature_categorical = [col for col in feature_categorical if col not in exclude_from_model]

print("Number of numeric predictors:", len(feature_numeric))
print("Number of categorical predictors:", len(feature_categorical))

print("\nNumerical predictors:")
print(feature_numeric)

print("\nCategorical predictors:")
print(feature_categorical)

Number of numeric predictors: 32
Number of categorical predictors: 11

Numerical predictors:
['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet', 'LotSizeAcres', 'LotSizeArea', 'YearBuilt', 'GarageSpaces', 'ParkingTotal', 'Latitude', 'Longitude', 'Stories', 'AssociationFee', 'MainLevelBedrooms', 'home_age', 'log_LivingArea', 'log_LotSizeSquareFeet', 'LivingArea_missing_flag', 'BedroomsTotal_missing_flag', 'BathroomsTotalInteger_missing_flag', 'LotSizeSquareFeet_missing_flag', 'LotSizeAcres_missing_flag', 'LotSizeArea_missing_flag', 'YearBuilt_missing_flag', 'GarageSpaces_missing_flag', 'ParkingTotal_missing_flag', 'Latitude_missing_flag', 'Longitude_missing_flag', 'Stories_missing_flag', 'AssociationFee_missing_flag', 'MainLevelBedrooms_missing_flag', 'home_age_missing_flag']

Categorical predictors:
['CountyOrParish', 'City', 'PostalCode', 'HighSchoolDistrict', 'Flooring', 'AttachedGarageYN', 'Levels', 'PoolPrivateYN', 'NewConstructionYN', 'ViewYN', 'Fireplace

In [5]:
X_train_raw = train_df[feature_numeric + feature_categorical].copy()
X_test_raw = test_df[feature_numeric + feature_categorical].copy()

y_train = train_df["ClosePrice"].copy()
y_test = test_df["ClosePrice"].copy()

print("X_train_raw shape:", X_train_raw.shape)
print("X_test_raw shape:", X_test_raw.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train_raw shape: (128568, 43)
X_test_raw shape: (11914, 43)
y_train shape: (128568,)
y_test shape: (11914,)


In [6]:
from sklearn.preprocessing import OneHotEncoder 
from sklearn.compose import ColumnTransformer

for col in feature_categorical:
    # converts everything to string 
    X_train_raw[col] = X_train_raw[col].astype(str)
    X_test_raw[col] = X_test_raw[col].astype(str)

try:
    encoder = OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=True
    )
except TypeError:
    encoder = OneHotEncoder(
        handle_unknown='ignore',
        sparse=True
    )


In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", feature_numeric), # Don't change these numeric column, just keep them
        ("cat", encoder, feature_categorical) # onehotencode for categorial features
    ],
    remainder='drop'
)

X_train_processed = preprocessor.fit_transform(X_train_raw)
X_test_processed = preprocessor.transform(X_test_raw)

In [8]:
print("Raw X_train shape:", X_train_raw.shape)
print("Processed X_train shape:", X_train_processed.shape)
print("Raw X_test shape:", X_test_raw.shape)
print("Processed X_test shape:", X_test_processed.shape)
print("Data type:", type(X_train_processed))

Raw X_train shape: (128568, 43)
Processed X_train shape: (128568, 3889)
Raw X_test shape: (11914, 43)
Processed X_test shape: (11914, 3889)
Data type: <class 'scipy.sparse._csr.csr_matrix'>


#### **Train Decision Tree**

In [9]:
from sklearn.tree import DecisionTreeRegressor

decision_tree = DecisionTreeRegressor(
    max_depth=12,
    min_samples_leaf=50,
    random_state=42
)

In [10]:
decision_tree.fit(X_train_processed, y_train)

dt_pred = decision_tree.predict(X_test_processed)


print("Decision Tree model trained.")
print("Number of predictions:", len(dt_pred))
print("First 5 predictions:", dt_pred[:5])

Decision Tree model trained.
Number of predictions: 11914
First 5 predictions: [ 910625.6473167   530825.41605839  360148.25200918 1699969.04615385
  739257.04964871]


In [11]:
dt_r2 = r2_score(y_test, dt_pred)
dt_mae = mean_absolute_error(y_test, dt_pred)
dt_rmse = np.sqrt(mean_squared_error(y_test, dt_pred))

print("Decision Tree Results")
print("R²:", dt_r2)
print("MAE:", dt_mae)
print("RMSE:", dt_rmse)

Decision Tree Results
R²: 0.767819687493566
MAE: 249688.93182258972
RMSE: 436461.49968683365


In [12]:
dt_results = []

for max_depth in [6, 8, 10, 12, 15, 18]:
    for min_samples_leaf in [20, 50, 100, 200]:
        model = DecisionTreeRegressor(
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )
        
        model.fit(X_train_processed, y_train)
        pred = model.predict(X_test_processed)
        
        r2 = r2_score(y_test, pred)
        mae = mean_absolute_error(y_test, pred)
        rmse = np.sqrt(mean_squared_error(y_test, pred))
        
        dt_results.append({
            "Model": "Decision Tree",
            "max_depth": max_depth,
            "min_samples_leaf": min_samples_leaf,
            "R2": r2,
            "MAE": mae,
            "RMSE": rmse
        })

dt_tuning_results = pd.DataFrame(dt_results)
dt_tuning_results.sort_values("R2", ascending=False).head(10)

,Model,max_depth,min_samples_leaf,R2,MAE,RMSE
20,Decision Tree,18,20,0.814337,211438.693273,390298.038735
16,Decision Tree,15,20,0.800382,223125.039107,404699.555558
21,Decision Tree,18,50,0.799606,221165.193494,405486.061425
17,Decision Tree,15,50,0.790254,229957.846291,414839.324568
22,Decision Tree,18,100,0.779992,234351.936635,424866.735634
18,Decision Tree,15,100,0.775644,238754.178830,429044.358673
12,Decision Tree,12,20,0.773300,247224.897013,431280.053932
13,Decision Tree,12,50,0.767820,249688.931823,436461.499687
14,Decision Tree,12,100,0.756957,254642.375918,446554.550415
8,Decision Tree,10,20,0.742458,271500.789764,459681.808030


#### **Random Forest**

In [15]:
from sklearn.ensemble import RandomForestRegressor

random_forest = RandomForestRegressor(
    n_estimators=50,
    max_depth=24,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1
)


random_forest.fit(X_train_processed, y_train)

rf_pred = random_forest.predict(X_test_processed)

print("Random Forest model trained.")
print("Number of predictions:", len(rf_pred))
print("First 5 predictions:", rf_pred[:5])

Random Forest model trained.
Number of predictions: 11914
First 5 predictions: [ 809508.39089275  541208.72084892  314012.45691043 1699505.64635634
  699332.55233804]


In [17]:
rf_results = []

for n_estimators in [30, 50]:
    for max_depth in [12, 18, 24]:
        for min_samples_leaf in [10, 20, 50]:
            model = RandomForestRegressor(
                n_estimators=n_estimators,
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                random_state=42,
                n_jobs=-1
            )
            
            model.fit(X_train_processed, y_train)
            pred = model.predict(X_test_processed)
            
            r2 = r2_score(y_test, pred)
            mae = mean_absolute_error(y_test, pred)
            rmse = np.sqrt(mean_squared_error(y_test, pred))
            
            rf_results.append({
                "Model": "Random Forest",
                "n_estimators": n_estimators,
                "max_depth": max_depth,
                "min_samples_leaf": min_samples_leaf,
                "R2": r2,
                "MAE": mae,
                "RMSE": rmse
            })

rf_tuning_results = pd.DataFrame(rf_results)
rf_tuning_results.sort_values("R2", ascending=False).head(10)

,Model,n_estimators,max_depth,min_samples_leaf,R2,MAE,RMSE
15,Random Forest,50,24,10,0.865646,173669.729391,332015.107951
6,Random Forest,30,24,10,0.865085,174160.276901,332708.359046
12,Random Forest,50,18,10,0.859977,181143.335899,338947.871623
3,Random Forest,30,18,10,0.859381,181643.995017,339668.962309
16,Random Forest,50,24,20,0.851572,183206.654457,348972.049542
7,Random Forest,30,24,20,0.851315,183742.704386,349275.027551
13,Random Forest,50,18,20,0.848602,187508.194859,352446.445680
4,Random Forest,30,18,20,0.848369,188009.406134,352718.093620
9,Random Forest,50,12,10,0.823711,217302.207579,380317.391846
0,Random Forest,30,12,10,0.823248,217655.534791,380816.183695


In [20]:
import numpy as np
import pandas as pd

final_comparison = pd.DataFrame({
    "Experiment": [
        "Original Linear Regression",
        "No-Outlier Linear Regression",
        "Original Random Forest",
        "No-Outlier Tuned Random Forest"
    ],
    "R2": [
        0.2871733970446031,
        0.8415249235130129,
        0.0340345465313624,
        0.865646
    ],
    "MAE": [
        452569.1626068585,
        220162.60581909068,
        371848.9678733415,
        173669.729391
    ],
    "RMSE": [
        1416770.3784853774,
        360590.0426259625,
        1649256.9231636084,
        332015.107951
    ]
})

final_comparison

,Experiment,R2,MAE,RMSE
0,Original Linear Regression,0.287173,452569.162607,1.416770e+06
1,No-Outlier Linear Regression,0.841525,220162.605819,3.605900e+05
2,Original Random Forest,0.034035,371848.967873,1.649257e+06
3,No-Outlier Tuned Random Forest,0.865646,173669.729391,3.320151e+05
